# Proyecto — Data Stream Processor

## Contexto (extremadamente importante): 

Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún `commit` he iniciar el proceso desde ese punto. 

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.


### Objetivo

Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases `ArrayStack` y `ArrayQueue` proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

En el método `__init__` debe aparecer tres atributos:
- **Queue:** registros que han llegado pero todavía no han sido procesados.
- **Stack:** historial de cambios realizados, para poder deshacer los cambios más recientes.
- **Lista:** como se encuentran los registros actualmente

Las clases `ArrayStack` y `ArrayQueue` ya están implementadas. **No debe implementarlas nuevamente ni modificarlas.**

In [6]:
import numbers
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.exceptions import Empty

In [7]:
class DataProcessor:
    class InvalidRecord(ValueError):
        """Excepción para registros inválidos."""

    def __init__(self):
        self._queue = ArrayQueue()
        self._history = ArrayStack()
        self._state = []

    def add(self, record):
        """Agrega un registro a la cola de pendientes. No lo procesa."""
        try:
            n = len(record)
        except TypeError:
            raise DataProcessor.InvalidRecord(
                "El registro debe ser una tupla (sensor, variable, value)."
            )
        if n != 3:
            raise DataProcessor.InvalidRecord(
                "El registro debe tener exactamente 3 componentes."
            )
        sensor, variable, value = record
        if isinstance(value, bool) or not isinstance(value, numbers.Number):
            raise DataProcessor.InvalidRecord("El 'value' del registro debe ser numérico.")
        self._queue.enqueue((sensor, variable, value))

    def _find_index(self, sensor, variable):
        """Busca (sensor, variable) en el estado actual. -1 si no existe. O(n)."""
        for i in range(len(self._state)):
            s, v, _ = self._state[i]
            if s == sensor and v == variable:
                return i
        return -1

    def process_next(self):
        """Procesa el registro más antiguo de la cola y actualiza el estado."""
        record = self._queue.dequeue()  # Empty si la cola está vacía
        sensor, variable, value = record

        idx = self._find_index(sensor, variable)
        if idx == -1:
            previous_value = None
            existed_before = False
            self._state.append(record)
        else:
            previous_value = self._state[idx][2]
            existed_before = True
            self._state[idx] = record

        self._history.push((record, previous_value, existed_before))
        return record

    def undo(self):
        """Deshace el último cambio aplicado por process_next(). LIFO."""
        record, previous_value, existed_before = self._history.pop()  # Empty si vacío
        sensor, variable, _ = record

        idx = self._find_index(sensor, variable)
        if existed_before:
            self._state[idx] = (sensor, variable, previous_value)
        else:
            del self._state[idx]

    def current_value(self, sensor, variable):
        """Devuelve el valor actual asociado con la combinación sensor/variable."""
        for s, v, value in self._state:
            if s == sensor and v == variable:
                return value
        raise KeyError((sensor, variable))

    def pending(self):
        """Número de registros que aún esperan ser procesados."""
        return len(self._queue)

In [8]:
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)

assert p.pending() == 3

p.process_next()
assert p.current_value("S01", "temperature") == 20

p.process_next()
assert p.current_value("S01", "temperature") == 25

p.process_next()
assert p.current_value("S01", "temperature") == 25
assert p.current_value("S01", "humidity") == 60

p.undo()
assert all(not (s == "S01" and v == "humidity") for s, v, _ in p._state)

p.undo()
assert p.current_value("S01", "temperature") == 20

p.undo()
assert p._state == []

print("Flujo A → B → C validado correctamente.")

Flujo A → B → C validado correctamente.


# INICIO DE PRUEBAS

In [9]:
#pending() sobre un procesador vacío
empty = DataProcessor()
assert empty.pending() == 0


In [10]:
#Agregar un registro
p1 = DataProcessor()
p1.add(("S01", "temperature", 20))
assert p1.pending() == 1

In [11]:
#Agregar varios registros
p2 = DataProcessor()
for record in [("S01", "temperature", 25), ("S01", "humidity", 60), ("S02", "pressure", 1013)]:
    p2.add(record)
assert p2.pending() == 3

In [12]:
#Verificar procesamiento FIFO
fifo = DataProcessor()
for record in [("S01", "temperature", 20), ("S01", "temperature", 25), ("S01", "humidity", 60)]:
    fifo.add(record)

first = fifo.process_next()
assert first == ("S01", "temperature", 20)
second = fifo.process_next()
assert second == ("S01", "temperature", 25)
assert fifo.pending() == 1

In [13]:
#Procesar un registro
single = DataProcessor()
single.add(("S03", "temperature", 18.5))
assert single.process_next() == ("S03", "temperature", 18.5)

In [14]:
#Procesar varios registros
multi = DataProcessor()
for record in [("A", "x", 1), ("A", "y", 2), ("B", "x", 3)]:
    multi.add(record)
assert multi.process_next() == ("A", "x", 1)
assert multi.process_next() == ("A", "y", 2)
assert multi.pending() == 1

In [15]:
#Actualizar una variable existente
upd = DataProcessor()
upd.add(("S01", "temperature", 20))
upd.process_next()
upd.add(("S01", "temperature", 25))
upd.process_next()
assert upd.current_value("S01", "temperature") == 25

In [16]:
#Consultar el valor actual
check = DataProcessor()
check.add(("S01", "humidity", 60))
check.process_next()
assert check.current_value("S01", "humidity") == 60

try:
    check.current_value("S99", "temperature")
    raise AssertionError("Se esperaba KeyError")
except KeyError:
    pass

In [17]:
#Realizar un undo()
undo_case = DataProcessor()
undo_case.add(("S01", "temperature", 20))
undo_case.add(("S01", "temperature", 25))
undo_case.process_next()
undo_case.process_next()

# Después de procesar ambos, el valor actual debe quedar en 25.
assert undo_case.current_value("S01", "temperature") == 25

# Undo debe revertir el último procesamiento y recuperar el valor anterior.
undo_case.undo()
assert undo_case.current_value("S01", "temperature") == 20

In [18]:
#Realizar varios undo()
multi_undo = DataProcessor()
for value in [10, 20, 30]:
    multi_undo.add(("S01", "temperature", value))

multi_undo.process_next()
assert multi_undo.current_value("S01", "temperature") == 10

multi_undo.process_next()
assert multi_undo.current_value("S01", "temperature") == 20

multi_undo.process_next()
assert multi_undo.current_value("S01", "temperature") == 30

multi_undo.undo()
assert multi_undo.current_value("S01", "temperature") == 20

multi_undo.undo()
assert multi_undo.current_value("S01", "temperature") == 10

In [19]:
#Procesar cuando la Queue está vacía
empty_queue = DataProcessor()
try:
    empty_queue.process_next()
    raise AssertionError("Se esperaba Empty")
except Empty:
    pass

In [20]:
#Hacer undo() cuando el historial está vacío
empty_history = DataProcessor()
try:
    empty_history.undo()
    raise AssertionError("Se esperaba Empty")
except Empty:
    pass

In [21]:
#Deshacer la creación de un dato que antes no existía
new_value = DataProcessor()
new_value.add(("S05", "pressure", 1000))
new_value.process_next()
assert new_value.current_value("S05", "pressure") == 1000
new_value.undo()
try:
    new_value.current_value("S05", "pressure")
    raise AssertionError("Se esperaba KeyError")
except KeyError:
    pass

In [22]:
#Hacer varios cambios sobre la misma variable
same_var = DataProcessor()
for value in [10, 20, 30]:
    same_var.add(("S02", "temperature", value))
    same_var.process_next()

assert same_var.current_value("S02", "temperature") == 30

same_var.undo()
assert same_var.current_value("S02", "temperature") == 20

same_var.undo()
assert same_var.current_value("S02", "temperature") == 10

In [23]:
#Agregar un registro con formato incorrecto
invalid_record = DataProcessor()
try:
    invalid_record.add(("S01", "temperature"))
    raise AssertionError("Se esperaba InvalidRecord")
except DataProcessor.InvalidRecord:
    pass

In [24]:
#Agregar un registro cuyo valor no sea numérico
non_numeric = DataProcessor()
try:
    non_numeric.add(("S01", "temperature", "alto"))
    raise AssertionError("Se esperaba InvalidRecord")
except DataProcessor.InvalidRecord:
    pass

In [25]:
#Consultar un sensor/variable que nunca haya sido procesado
missing_pair = DataProcessor()
try:
    missing_pair.current_value("S99", "temperature")
    raise AssertionError("Se esperaba KeyError")
except KeyError:
    pass

# COMPLEJIDAD

1: add(record)

Complejidad: O(1) amortizado

Justificación:

Se valida la estructura de la tupla y el tipo numérico del valor.

Eso toma tiempo constante.

Luego se hace self._queue.enqueue(record), y la operación de cola es O(1) amortizado en ArrayQueue.

2: process_next()

Complejidad: O(n)

Justificación:

self._queue.dequeue() es O(1).

Pero luego se llama a _find_index(sensor, variable), que recorre _state buscando la combinación.

Si hay n elementos en _state, esa búsqueda puede recorrer todos.

La actualización del estado y el push al historial son O(1), pero no cambian el hecho de que la búsqueda lineal domina.

3: undo()

Complejidad: O(n)

Justificación:

self._history.pop() es O(1).

Luego se vuelve a buscar la combinación en _state usando _find_index, que es O(n).

Si el elemento no existía antes, se hace del self._state[idx], y eliminar un elemento de una lista puede desplazar los siguientes, también costando 

O(n) en el peor caso.

4: pending()

Complejidad: O(1)

Justificación:

Solo retorna len(self._queue).

Obtener la longitud de una cola es una operación constante.
Resultado:

No depende del número de registros pendientes.

5: current_value(sensor, variable)

Complejidad: O(n)

Justificación:

Recorre la lista _state comparando cada (sensor, variable).

Si la clave buscada está al final o no existe, puede recorrer toda la lista.

# BONUS

In [26]:

class DataProcessor:
    class InvalidRecord(ValueError):
        """Excepción para registros inválidos."""

    def __init__(self):
        self._queue = ArrayQueue()
        self._history = ArrayStack()   # operaciones realizadas
        self._redo = ArrayStack()      # operaciones deshechas
        self._state = []

    def add(self, record):
        """Agrega un registro a la cola de pendientes. No lo procesa."""
        try:
            n = len(record)
        except TypeError:
            raise DataProcessor.InvalidRecord(
                "El registro debe ser una tupla (sensor, variable, value)."
            )

        if n != 3:
            raise DataProcessor.InvalidRecord(
                "El registro debe tener exactamente 3 componentes."
            )

        sensor, variable, value = record

        if isinstance(value, bool) or not isinstance(value, numbers.Number):
            raise DataProcessor.InvalidRecord(
                "El 'value' del registro debe ser numérico."
            )

        self._queue.enqueue((sensor, variable, value))

    def _find_index(self, sensor, variable):
        """Busca (sensor, variable) en el estado actual. Retorna -1 si no existe."""
        for i in range(len(self._state)):
            s, v, _ = self._state[i]
            if s == sensor and v == variable:
                return i
        return -1

    def process_next(self):
        """Procesa el registro más antiguo de la cola y actualiza el estado."""
        record = self._queue.dequeue()  # Empty si la cola está vacía
        sensor, variable, value = record

        idx = self._find_index(sensor, variable)

        if idx == -1:
            previous_value = None
            existed_before = False
            self._state.append(record)
        else:
            previous_value = self._state[idx][2]
            existed_before = True
            self._state[idx] = record

        self._history.push((record, previous_value, existed_before))

        # Al realizar una nueva operación ya no tiene sentido rehacer
        self._redo = ArrayStack()

        return record

    def undo(self):
        """Deshace el último cambio aplicado por process_next()."""
        record, previous_value, existed_before = self._history.pop()

        sensor, variable, _ = record
        idx = self._find_index(sensor, variable)

        if existed_before:
            self._state[idx] = (sensor, variable, previous_value)
        else:
            del self._state[idx]

        # Guardar para posible redo
        self._redo.push((record, previous_value, existed_before))

    def redo(self):
        """Reaplica el último cambio deshecho."""
        record, previous_value, existed_before = self._redo.pop()

        sensor, variable, value = record

        idx = self._find_index(sensor, variable)

        if idx == -1:
            self._state.append(record)
        else:
            self._state[idx] = record

        # Vuelve a quedar en el historial
        self._history.push((record, previous_value, existed_before))

        return record

    def current_value(self, sensor, variable):
        """Devuelve el valor actual asociado con sensor/variable."""
        for s, v, value in self._state:
            if s == sensor and v == variable:
                return value

        raise KeyError((sensor, variable))

    def pending(self):
        """Número de registros pendientes por procesar."""
        return len(self._queue)


## Estructuras utilizadas
ArrayStack para _history: guarda los cambios aplicados por process_next().

ArrayStack para _redo: guarda los cambios deshechos por undo().

ArrayQueue sigue siendo la cola de registros pendientes.

self._state sigue representando el estado actual.
### ¿Por qué estas estructuras?
Porque undo() y redo() deben seguir una lógica LIFO: el último cambio deshecho es el primero en rehacerse. Una pila es la estructura natural para eso. Además, al hacer un nuevo process_next() después de un undo(), se limpia _redo para evitar rehacer cambios que ya quedaron invalidados por un flujo nuevo.

In [27]:
from Proyecto1.evaluador import *

In [28]:
ejecutar(DataProcessor, estudiante="Nicolas Diaz Torres")

EVALUACION - Data Stream Processor
Estudiante : Nicolas Diaz Torres
Fecha      : 2026-09-21 20:07:07
Python     : 3.12.14 (Linux)
Evaluador  : v1.1
Clase      : DataProcessor (modulo __main__)

== Interfaz y restricciones ==

[PASS ] Define add, process_next, undo, pending y current_value
[PASS ] Se puede crear DataProcessor() sin argumentos
[PASS ] __init__ crea una ArrayQueue, un ArrayStack y una list
[PASS ] No sustituye Queue/Stack por deque u otra estructura
[PASS ] ArrayQueue y ArrayStack son las del curso (no reimplementadas)

   Interfaz y restricciones: 5/5 OK

== Pruebas obligatorias (1-17) ==

[PASS ] 1. pending() sobre un procesador vacio
[PASS ] 2. Agregar un registro
[PASS ] 3. Agregar varios registros
[PASS ] 4. Procesamiento FIFO
[PASS ] 5. Procesar un registro
[PASS ] 6. Procesar varios registros
[PASS ] 7. Actualizar una variable existente
[PASS ] 8. Consultar el valor actual
[PASS ] 9. Realizar un undo()
[PASS ] 10. Varios undo() consecutivos
[PASS ] 11. process_next